In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # 1 GPU: evita DataParallel multi-T4


# LoRA Qwen3-1.7B en GPU gratis (BTBbro)
1. Acelerador: GPU T4 (Kaggle: Settings → Accelerator → GPU; Colab: Entorno → GPU).
2. Subí `lora_train.jsonl`, `lora_val.jsonl`, `lora_test.jsonl` al explorador de archivos.
3. Ejecutar todo en orden. Dura ~4-5h. Los checkpoints quedan en `./lora-out`.
4. Al final descargá `predictions_test.csv` y traelo al repo para la 4ª curva.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>&1 | head -n 3

In [ ]:
%pip install -q transformers peft trl bitsandbytes accelerate scikit-learn 2>&1 | tail -n 1

In [ ]:
import json, glob
def load(name):
    cands = glob.glob(f"**/{name}.jsonl", recursive=True)
    cands += glob.glob(f"/kaggle/input/**/{name}.jsonl", recursive=True)
    p = cands[0]
    rows = [json.loads(l) for l in open(p, encoding="utf-8")]
    print(name, len(rows), "<-", p)
    return rows
train, val, test = load("lora_train"), load("lora_val"), load("lora_test")


In [ ]:
SYS = ("You are an educational BTC news classifier. Output EXACTLY two lines:\n"
       "ACTION: BUY or SELL or HOLD\nREASON: <=12 words")
def fmt(r, for_train=True):
    u = f"TITLE: {r['title']}\nBODY: {r.get('body','')}\nRespond with ACTION and REASON lines only."
    p = f"<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n{u}<|im_end|>\n<|im_start|>assistant\n"
    if for_train:
        p += f"ACTION: {r['label'].upper()}\nREASON: supervised training example<|im_end|>"
    return p
print(fmt(train[0])[:300])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODEL = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                            device_map="auto", trust_remote_code=True)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
base = prepare_model_for_kbit_training(base)
peft = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj"])
model = get_peft_model(base, peft)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer
def tok_fn(rows):
    return tok([fmt(r) for r in rows], truncation=True, max_length=256)
dtr = Dataset.from_list([{"text": fmt(r)} for r in train])
dtr = dtr.map(lambda b: tok(b["text"], truncation=True, max_length=256),
              batched=True, remove_columns=["text"])
args = SFTConfig(output_dir="./lora-out", num_train_epochs=2, per_device_train_batch_size=4,
                 gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=50,
                 save_steps=300, save_total_limit=3, seed=42, fp16=True,
                 report_to="none")
SFTTrainer(model=model, train_dataset=dtr, args=args).train()  # si se corta: .train(resume_from_checkpoint="./lora-out/checkpoint-N")


In [ ]:
import re
from sklearn.metrics import f1_score
ACT = re.compile(r"ACTION\s*:\s*(BUY|SELL|HOLD)", re.I)
def predict(rows):
    out = []
    model.eval()
    for r in rows:
        ids = tok(fmt(r, for_train=False), return_tensors="pt").to(model.device)
        with __import__("torch").no_grad():
            g = model.generate(**{k: v for k, v in ids.items() if k != "token_type_ids"},
                               max_new_tokens=30, do_sample=False, pad_token_id=tok.eos_token_id)
        txt = tok.decode(g[0][ids["input_ids"].shape[1]:])
        m = ACT.search(txt)
        out.append(m.group(1).lower() if m else "hold")
    return out
for name, rows in (("val", val), ("test", test)):
    p = predict(rows)
    y = [r["label"] for r in rows]
    print(name, "macro-F1=", round(f1_score(y, p, average="macro", zero_division=0), 3))
import csv
w = csv.writer(open("predictions_test.csv", "w", newline="", encoding="utf-8"))
w.writerow(["id", "action"])
w.writerows(zip([r["id"] for r in test], predict(test)))
print("predictions_test.csv OK")